# Tokenization Comparison

- Creator: Sebastian Birk (sebastian.birk@helmholtz-munich.de).
- Date of Creation: 05.06.2024
- Date of Last Modification: 11.06.2024

## 1. Setup

### 1.1 Import Libraries

In [ ]:
import warnings

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

### 1.2 Run Notebook Setup

In [ ]:
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

### 1.3 Configure Paths and Create Directories

In [ ]:
# Define paths
st_data_folder_path = "../datasets/st_data" # spatially resolved transcriptomics data
st_data_bronze_folder_path = f"{st_data_folder_path}/bronze"
st_data_silver_folder_path = f"{st_data_folder_path}/silver"
st_data_gold_folder_path = f"{st_data_folder_path}/gold"
tokenizer_folder_path = f"../datasets/tokenizer/{tokenizer}"
results_folder_path = "../datasets/results"

## 2. Tokenization Comparison

In [ ]:
# Define dataset
dataset_name = "nanostring_cosmx_human_brain"
#dataset_name = "xenium_human_brain_cancer"

In [ ]:
def capitalize_after_space(s):
    return " ".join(word.capitalize() for word in s.split("_"))

# Load data
tokenizer = "cell_graph_tokenizer"
df_gt = pd.read_csv(f"{results_folder_path}/{tokenizer}/{dataset_name}/clf_scores_scgpt.csv")
tokenizer = "cell_neighborhood_tokenizer"
df_nt = pd.read_csv(f"{results_folder_path}/{tokenizer}/{dataset_name}/clf_scores_scgpt.csv")
df = pd.concat([df_gt, df_nt], ignore_index=True)

# Only keep test scores
df = df[df["clf_category"] == "test"]

# Rename token types
df["token_type"] = df["token_type"].replace(
    {"cell": "Index Cell Tokens", "neighbor_cells": "Neighbor Cell Tokens"})

# Rename classification tasks
df["clf_task"] = df["clf_task"].replace(
    {"cell_types": "Cell Type Classification", "niche_types": "Niche Type Classification"})

# Create a combined hue for clf_task and clf_category
df["token_task"] = df["token_type"] + " - " + df["clf_task"]

# Create subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 7), sharey=True)

# Plot for Cell Graph Tokenizer
sns.barplot(
    y="score", x="normalization", hue="token_task", data=df[df["tokenizer"] == "cell_neighborhood_tokenizer"], ci="sd", 
    palette="Set3", orient="v", width=0.6, ax=axes[0]
)
axes[0].set_title("Cell Neighborhood Tokenizer", fontsize=16, fontweight="bold")
axes[0].set_ylabel("Accuracy")
axes[0].set_xlabel("Normalization Method")
axes[0].legend_.remove()  # Remove legend from the first subplot
axes[0].grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.7)
axes[0].tick_params(axis='x', rotation=90)  # Rotate x-axis labels

# Plot for Cell Neighborhood Tokenizer
sns.barplot(
    y="score", x="normalization", hue="token_task", data=df[df["tokenizer"] == "cell_graph_tokenizer"], ci="sd", 
    palette="Set3", orient="v", width=0.6, ax=axes[1]
)
axes[1].set_title("Cell Graph Tokenizer", fontsize=16, fontweight="bold")
axes[1].set_ylabel("Accuracy")
axes[1].set_xlabel("Normalization Method")
axes[1].grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.7)
axes[1].tick_params(axis='x', rotation=90)  # Rotate x-axis labels
axes[1].legend_.remove()  # Remove legend from the second subplot

# Add a single legend below both plots
handles, labels = axes[1].get_legend_handles_labels()
fig.legend(handles, labels, title="Token Type - Classification Task", bbox_to_anchor=(0.5, -0.05), loc="upper center", ncol=2)

# Overall title for the figure
fig.suptitle("Linear Probing Accuracy with Top 10 Ranked Tokens (scGPT Embedded)", fontsize=20, fontweight="bold", y=0.9)

# Adjust layout
fig.tight_layout(rect=[0, -0.05, 1., 0.9])  # Adjust layout to make room for the legend

plt.savefig(f"{results_folder_path}/{dataset_name}/clf_scores_scgpt.svg", bbox_inches="tight")
plt.show()